In [ ]:
import cv2
import numpy as np
import os
from skimage.feature import hog
from scipy.spatial.distance import cosine
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

# ==========================================
# 1. EXTRACT STRUCTURE (HOG FINGERPRINT)
# ==========================================
def extract_structural_fingerprint(image, target_size=(250, 400)):
    """
    Extracts purely the structural edges/layout of the card using
    Histogram of Oriented Gradients (HOG).
    """
    # Resize to standard aspect ratio
    resized = cv2.resize(image, target_size)
    # Convert to grayscale (we only care about structure, not color)
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)

    # Extract HOG features
    features = hog(gray, orientations=9, pixels_per_cell=(16, 16),
                   cells_per_block=(2, 2), transform_sqrt=True, block_norm="L2-Hys")

    return features

# ==========================================
# 2. SAVE THE GOLD STANDARD TEMPLATE
# ==========================================
def save_master_template(base_image_path, save_path="master_hog_template.npy"):
    """Extracts the HOG vector from the perfect card and saves it to disk."""
    base_image = cv2.imread(base_image_path)
    if base_image is None:
        raise ValueError(f"Could not load master image at {base_image_path}")

    master_vector = extract_structural_fingerprint(base_image)
    np.save(save_path, master_vector)
    print(f"✅ Master Template successfully saved to {save_path}")

    return master_vector

# ==========================================
# 3. INFERENCE (COSINE DISTANCE SCORING)
# ==========================================
def calculate_structural_deviation(test_image_path, master_vector):
    """
    Compares a new upload to the master template using Cosine Distance.
    Returns a pure distance score (0.0 = Identical, 1.0 = Completely Different).
    """
    test_image = cv2.imread(test_image_path)
    if test_image is None:
        raise ValueError(f"Could not load test image at {test_image_path}")

    test_vector = extract_structural_fingerprint(test_image)

    # Cosine distance calculates the angle between the two structural arrays
    distance_score = cosine(master_vector, test_vector)

    return distance_score


# ==========================================
# MAIN EXECUTION & DYNAMIC METRIC TUNING
# ==========================================
if __name__ == "__main__":

    BASE_IMAGE_PATH = "adhaar card.png"
    MODEL_PATH = "master_hog_template.npy"

    # --- DUMMY IMAGE GENERATOR (To make the script run out-of-the-box) ---
    if not os.path.exists(BASE_IMAGE_PATH):
        print(f"[INFO] Creating a dummy master image at '{BASE_IMAGE_PATH}'...")
        dummy_img = np.ones((500, 800, 3), dtype=np.uint8) * 200 # Blank gray card
        cv2.putText(dummy_img, "AADHAAR", (300, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0,0,0), 3)
        cv2.rectangle(dummy_img, (50, 150), (200, 350), (100, 100, 100), -1) # Dummy Photo
        cv2.putText(dummy_img, "Name: John Doe", (250, 200), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,0), 2)
        cv2.imwrite(BASE_IMAGE_PATH, dummy_img)
    # ----------------------------------------------------------------------

    # 1. Save the Master Template (This represents your perfectly verified original ID)
    master_vec = save_master_template(BASE_IMAGE_PATH, MODEL_PATH)
    # ---------------------------------------------------------
    # 2. GENERATE TEST DATA (Updated with Variance)
    # ---------------------------------------------------------
    print("\nGenerating Synthetic Test Sets...")
    y_true = []
    y_scores = []
    base_img = cv2.imread(BASE_IMAGE_PATH)

    # A. Generate 20 "Legitimate" Test Cards
    for _ in range(20):
        img_copy = base_img.copy()
        angle = np.random.uniform(-2, 2)
        M = cv2.getRotationMatrix2D((img_copy.shape[1]/2, img_copy.shape[0]/2), angle, 1.0)
        img_copy = cv2.warpAffine(img_copy, M, (img_copy.shape[1], img_copy.shape[0]))

        test_vec = extract_structural_fingerprint(img_copy)
        y_scores.append(cosine(master_vec, test_vec))
        y_true.append(0) # 0 = Legit

    # B. Generate 20 "Fraud" Test Cards (Now with random box sizes!)
    for _ in range(20):
        fake_img = base_img.copy()

        # Add random variance to the black box so every fake is unique
        box_width = np.random.randint(200, 400)
        box_height = np.random.randint(150, 300)
        cv2.rectangle(fake_img, (100, 100), (100 + box_width, 100 + box_height), (0, 0, 0), -1)

        test_vec = extract_structural_fingerprint(fake_img)
        y_scores.append(cosine(master_vec, test_vec))
        y_true.append(1) # 1 = Fraud

    # ---------------------------------------------------------
    # 3. DYNAMIC METRIC TUNING (Updated Operator)
    # ---------------------------------------------------------
    print("\n--- RUNNING METRIC EVALUATION ---")

    fpr, tpr, thresholds = roc_curve(y_true, y_scores)

    # Find the optimal decision boundary
    optimal_idx = np.argmax(tpr - fpr)
    OPTIMAL_THRESHOLD = thresholds[optimal_idx]

    print(f"[AUTO-TUNING] Mathematically Optimal Distance Threshold: {OPTIMAL_THRESHOLD:.4f}")

    # 🌟 THE FIX: Changed > to >=
    y_pred = [1 if score >= OPTIMAL_THRESHOLD else 0 for score in y_scores]

    # Print Academic Metrics
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall   : {recall_score(y_true, y_pred):.4f}")
    print(f"F1-Score : {f1_score(y_true, y_pred):.4f}")
    print(f"ROC-AUC  : {roc_auc_score(y_true, y_scores):.4f}")

    print("\nConfusion Matrix [TN, FP] / [FN, TP]:")
    print(confusion_matrix(y_true, y_pred))

    # ---------------------------------------------------------
    # 4. THE ULTIMATE SINGLE IMAGE TEST
    # ---------------------------------------------------------
    print("\n--- SINGLE IMAGE INFERENCE TEST ---")

    # Load the saved model (the Numpy vector) from disk just to prove the full pipeline works
    loaded_master_vec = np.load(MODEL_PATH)

    # Test the original image against its own master template
    original_score = calculate_structural_deviation(BASE_IMAGE_PATH, loaded_master_vec)

    print(f"Original Image Fraud Score: {original_score:.6f} (Exactly 0.0 is Perfect)")
    if original_score <= OPTIMAL_THRESHOLD:
        print("Status: ✅ APPROVED (Structurally matches master template)")
    else:
        print("Status: 🛑 REJECTED (Structural deviation exceeds safe threshold)")

✅ Master Template successfully saved to master_hog_template.npy

Generating Synthetic Test Sets...

--- RUNNING METRIC EVALUATION ---
[AUTO-TUNING] Mathematically Optimal Distance Threshold: 0.2226
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1-Score : 1.0000
ROC-AUC  : 1.0000

Confusion Matrix [TN, FP] / [FN, TP]:
[[20  0]
 [ 0 20]]

--- SINGLE IMAGE INFERENCE TEST ---
Original Image Fraud Score: 0.000000 (Exactly 0.0 is Perfect)
Status: ✅ APPROVED (Structurally matches master template)


In [ ]:
import os
import cv2
import numpy as np
from skimage.feature import hog
from scipy.spatial.distance import cosine

# ==========================================
# 1. EXTRACT STRUCTURE (HOG FINGERPRINT)
# ==========================================
def extract_structural_fingerprint(image, target_size=(250, 400)):
    """Extracts purely the structural edges/layout of the card."""
    resized = cv2.resize(image, target_size)
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)

    features = hog(gray, orientations=9, pixels_per_cell=(16, 16),
                   cells_per_block=(2, 2), transform_sqrt=True, block_norm="L2-Hys")
    return features

# ==========================================
# 2. SINGLE IMAGE INFERENCE PIPELINE
# ==========================================
def analyze_single_image(image_path, master_npy_path, threshold=0.32):
    """
    Analyzes a single uploaded image against the master template.
    """
    print(f"\n[INFO] Analyzing Upload: '{image_path}'")

    # 1. Check if files exist
    if not os.path.exists(master_npy_path):
        print(f"[ERROR] Master template not found at {master_npy_path}. Run training first.")
        return

    if not os.path.exists(image_path):
        print(f"[ERROR] Uploaded image not found at {image_path}.")
        return

    # 2. Load the pre-calculated Master Vector
    master_vec = np.load(master_npy_path)

    # 3. Read the uploaded image
    img = cv2.imread(image_path)
    if img is None:
        print(f"[ERROR] Could not decode the image file.")
        return

    # 4. Extract and Compare
    test_vec = extract_structural_fingerprint(img)
    distance_score = cosine(master_vec, test_vec)

    # 5. Print the Final Verdict
    print("\n" + "="*45)
    print("         STRUCTURAL FORGERY REPORT          ")
    print("="*45)
    print(f"Applied Distance Threshold : >= {threshold:.4f}")
    print(f"Calculated Deviation Score : {distance_score:.4f}")
    print("-" * 45)

    if distance_score >= threshold:
        print("Status: 🛑 REJECTED (High Forgery Probability)")
        print("Reason: Structural layout deviates significantly from the authentic template.")
    else:
        print("Status: ✅ APPROVED (Structurally Authentic)")

    print("="*45)

    return distance_score

# ==========================================
# RUN THE SCRIPT
# ==========================================
if __name__ == "__main__":

    # Configuration
    TEST_IMAGE_PATH = "2.jpg"
    MASTER_NPY_FILE = "master_hog_template.npy"
    DISTANCE_THRESHOLD = 0.6

    # --- DUMMY IMAGE GENERATOR (So the script runs immediately) ---
    if not os.path.exists(TEST_IMAGE_PATH):
        print(f"[SETUP] Creating a dummy test image at '{TEST_IMAGE_PATH}'...")
        # We will create a "Fake" looking card with a huge black box
        dummy_img = np.ones((500, 800, 3), dtype=np.uint8) * 200
        cv2.putText(dummy_img, "AADHAAR", (300, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0,0,0), 3)
        cv2.rectangle(dummy_img, (50, 150), (400, 450), (0, 0, 0), -1) # Huge anomaly
        cv2.imwrite(TEST_IMAGE_PATH, dummy_img)
    # --------------------------------------------------------------

    # Run the Inference
    analyze_single_image(TEST_IMAGE_PATH, MASTER_NPY_FILE, DISTANCE_THRESHOLD)


[INFO] Analyzing Upload: '2.jpg'

         STRUCTURAL FORGERY REPORT          
Applied Distance Threshold : >= 0.6000
Calculated Deviation Score : 0.4710
---------------------------------------------
Status: ✅ APPROVED (Structurally Authentic)
